In [1]:
import pandas as pd
import numpy as np
import re 
import html  
import unicodedata


In [20]:
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, confusion_matrix


In [3]:
ceas = pd.read_csv('datasets/CEAS_08.csv')

In [4]:
ceas.info()

<class 'pandas.DataFrame'>
RangeIndex: 39154 entries, 0 to 39153
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   sender    39154 non-null  str  
 1   receiver  38692 non-null  str  
 2   date      39154 non-null  str  
 3   subject   39126 non-null  str  
 4   body      39154 non-null  str  
 5   label     39154 non-null  int64
 6   urls      39154 non-null  int64
dtypes: int64(2), str(5)
memory usage: 66.3 MB


In [5]:
ceas['label'].value_counts()

label
1    21842
0    17312
Name: count, dtype: int64

In [7]:
ceas[ceas['label']==0].head()

,sender,receiver,date,subject,body,label,urls
3,Michael Parker <ivqrnai@pobox.com>,SpamAssassin Dev <xrh@spamassassin.apache.org>,"Tue, 05 Aug 2008 17:31:20 -0600",Re: svn commit: r619753 - in /spamassassin/tru...,Would anyone object to removing .so from this ...,0,1
8,qydlqcws-iacfym@issues.apache.org,xrh@spamassassin.apache.org,"Tue, 05 Aug 2008 15:31:03 -0800",[Bug 5780] URI processing turns uuencoded stri...,http://issues.apache.org/SpamAssassin/show_bug...,0,1
15,Racing <uqyrmo@sailing.ie>,user5@gvc.ceas-challenge.cc,"Wed, 06 Aug 2008 00:31:14 +0100",RE: Trial IRC Certificate Application,"\nPlelim,\n\nJust to remind you that if a cert...",0,1
18,Aaron Kulkis <cmiqlkx91@hotpop.com>,opensuse <wkilxloc@opensuse.org>,"Tue, 05 Aug 2008 15:50:37 -0500","Re: [opensuse] Why can't I use ""shutdown now"" ...",Carlos E. R. wrote: > -----BEGIN PGP SIGNED ME...,0,1
19,Aaron Kulkis <cmiqlkx91@hotpop.com>,opensuse <wkilxloc@opensuse.org>,"Tue, 05 Aug 2008 16:31:41 -0500",Re: Fwd: [opensuse] Re: openSUSE Boxed Editions,Steve Jacobs wrote: > ---------- Forwarded mes...,0,1


In [8]:
ceas[ceas['label']==1].head()

,sender,receiver,date,subject,body,label,urls
0,Young Esposito <Young@iworld.de>,user4@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 16:31:02 -0700",Never agree to be a loser,"Buck up, your troubles caused by small dimensi...",1,1
1,Mok <ipline's1983@icable.ph>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 18:31:03 -0500",Befriend Jenna Jameson,\nUpgrade your sex and pleasures with these te...,1,1
2,Daily Top 10 <Karmandeep-opengevl@universalnet...,user2.9@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 20:28:00 -1200",CNN.com Daily Top 10,>+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+=+...,1,1
4,Gretchen Suggs <externalsep1@loanofficertool.com>,user2.2@gvc.ceas-challenge.cc,"Tue, 05 Aug 2008 19:31:21 -0400",SpecialPricesPharmMoreinfo,\nWelcomeFastShippingCustomerSupport\nhttp://7...,1,1
5,Caroline Aragon <dwthaidomainnamesm@thaidomain...,user7-ext5@gvc.ceas-challenge.cc,"Wed, 06 Aug 2008 05:31:22 +0600",From Caroline Aragon,\n\n\n\n\nYo wu urS mo ou go rc ebo eForM rgi ...,1,0


In [ ]:
# keeping only the columns that are needed
needed_cols = ["subject", "body", "sender", "label"]
df = ceas[needed_cols].copy()


In [ ]:
# filling in the missing text fields with blank strings
df["subject"] = df["subject"].fillna("")
df["body"] = df["body"].fillna("")
df["sender"] = df["sender"].fillna("")

# removing exact duplicate subject+body pairs
df = df.drop_duplicates(subset=["subject", "body"]).reset_index(drop=True)

print("\nAfter cleaning:", df.shape)
print(df["label"].value_counts())



After cleaning: (39154, 4)
label
1    21842
0    17312
Name: count, dtype: int64


In [ ]:
# very important text preprocessing step for the body+header field


def strip_html(text: str) -> str:    #removing HTML tags
    text = re.sub(r"<[^>]+>", " ", text)
    return text

def remove_urls(text: str) -> str:    #removing any url links that may affect the emotion labelling
    text = re.sub(r"http[s]?://\S+", " ", text)
    text = re.sub(r"www\.\S+", " ", text)
    return text

def remove_emails(text: str) -> str:      #removing email addresses
    text = re.sub(r"\b[\w\.-]+@[\w\.-]+\.\w+\b", " ", text)
    return text

def normalize_whitespace(text: str) -> str:   #making multiple whitespaces into a single one
    return re.sub(r"\s+", " ", text).strip()

def remove_reply_markers(text: str) -> str:     # removing repeated quoted markers and common forwarded prefixes
    text = re.sub(r"(^|\n)\s*>+.*", " ", text)
    text = re.sub(r"forwarded message", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"original message", " ", text, flags=re.IGNORECASE)
    return text

def keep_letters_only(text: str) -> str: #removing any non letter characters and make letters lowercase
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    return text

def preprocess_text(text: str) -> str:   #integrating everything into one function
    text = str(text)
    text = strip_html(text)
    text = remove_urls(text)
    text = remove_emails(text)
    text = remove_reply_markers(text)
    text = keep_letters_only(text)
    text = normalize_whitespace(text)
    return text

In [ ]:
# applying preprocessing function on relevant attributes
df["subject_clean"] = df["subject"].apply(preprocess_text)
df["body_clean"] = df["body"].apply(preprocess_text)

# combined text
df["text_clean"] = (df["subject_clean"] + " " + df["body_clean"]).str.strip()


In [15]:
df["text_clean"].duplicated().value_counts()

text_clean
False    32652
True      6502
Name: count, dtype: int64

In [ ]:
df = df.drop_duplicates(subset=['text_clean']).reset_index(drop=True) #removing duplicates 

In [17]:
df["text_clean"].duplicated().value_counts()

text_clean
False    32652
Name: count, dtype: int64

In [ ]:

#defining lexicon dictionary for different intents and emotions

lexicons = {
    # Intent
    "urgency": [
        "urgent", "immediately", "immediate", "now", "asap", "today",
        "deadline", "expires", "expiring", "limited", "hurry", "soon"
    ],
    "authority": [
        "admin", "administrator", "official", "department", "manager",
        "director", "team", "support", "service", "office"
    ],
    "request_action": [
        "please", "kindly", "assist", "help", "request", "confirm",
        "review", "respond", "reply", "contact", "send", "complete"
    ],
    "promotion_marketing": [
        "offer", "deal", "discount", "sale", "free", "bonus",
        "special", "promotion", "limited offer", "save"
    ],
    "social_connection": [
        "friend", "meet", "connect", "join", "befriend",
        "relationship", "partner", "community"
    ],
    "information_transactional": [
        "report", "statement", "notice", "update", "details",
        "information", "summary", "record", "status"
    ],
    "technical_discussion": [
        "commit", "patch", "bug", "issue", "version", "code",
        "system", "mailing", "list", "discussion", "developer"
    ],

    # Emotion
    "fear": [
        "risk", "warning", "problem", "danger", "failure",
        "error", "trouble", "concern", "threat"
    ],
    "greed": [
        "win", "prize", "reward", "money", "cash",
        "earn", "profit", "wealth", "rich"
    ],
    "curiosity": [
        "discover", "secret", "hidden", "unknown",
        "revealed", "surprising", "exclusive"
    ],
    "trust": [
        "trusted", "official", "secure", "verified", "safe",
        "reliable", "approved"
    ],
    "excitement": [
        "amazing", "incredible", "fantastic", "exclusive",
        "exciting", "wonderful", "great"
    ],
    "sexual_pleasure": [
        "sex", "pleasure", "adult", "hot", "dating",
        "intimate", "desire", "romance"
    ],
    "neutral_formal": [
        "regards", "sincerely", "thanks", "attached",
        "discussion", "proposal", "meeting", "committee"
    ]
}

# converting all lexicon terms to lowercase
lexicons = {
    k: [term.lower() for term in v]
    for k, v in lexicons.items()
}


In [ ]:
#extraction of all features


def tokenize(text: str): #splitting cleaned text into words
    if not text:
        return []
    return text.split()  

def count_term_matches(tokens, lexicon_terms):  #using a counter to count lexicon matches to words
    # counts both single and multi word lexicon matches
    token_count = 0
    text = " ".join(tokens)

    single_terms = [t for t in lexicon_terms if " " not in t]   #defining for single word
    multi_terms = [t for t in lexicon_terms if " " in t]    #defining for multiword

    token_counter = Counter(tokens)

    # single word counts
    for term in single_terms:
        token_count += token_counter.get(term, 0)

    # multi word counts
    for phrase in multi_terms:
        token_count += len(re.findall(rf"\b{re.escape(phrase)}\b", text))

    return token_count

def compute_category_features(text: str, lexicon_terms):
    tokens = tokenize(text)
    n_tokens = len(tokens)

    raw_count = count_term_matches(tokens, lexicon_terms)  #defining the actual count of matches
    binary_presence = int(raw_count > 0)  # a binary variable to see if any category even appears
    normalized_count = raw_count / n_tokens if n_tokens > 0 else 0.0   #noirmalizing the count against the 
                                                                       #length of the email to minimize the effect of email length

    return raw_count, binary_presence, normalized_count

def extract_lexicon_features(row, lexicons_dict):
    features = {}  #dictionary for the features where the format of the keyss is "(category)_(text_field)_(feature_type)"
    # category refers to any of the intents/emotions
    # text_field refers to one of subject/body/combined
    # feature_type refers to one of raw/bin/norm
    

    #doing for all three of subject and body alone as well as them together for flexibility later on
    subject_text = row["subject_clean"]
    body_text = row["body_clean"]
    combined_text = row["text_clean"]

    
    #computing category features for each of the three
    for category, terms in lexicons_dict.items():
        # subject
        s_raw, s_bin, s_norm = compute_category_features(subject_text, terms)
        features[f"{category}_subject_raw"] = s_raw
        features[f"{category}_subject_bin"] = s_bin
        features[f"{category}_subject_norm"] = s_norm

        # body
        b_raw, b_bin, b_norm = compute_category_features(body_text, terms)
        features[f"{category}_body_raw"] = b_raw
        features[f"{category}_body_bin"] = b_bin
        features[f"{category}_body_norm"] = b_norm

        # combined
        c_raw, c_bin, c_norm = compute_category_features(combined_text, terms)
        features[f"{category}_combined_raw"] = c_raw
        features[f"{category}_combined_bin"] = c_bin
        features[f"{category}_combined_norm"] = c_norm

    return pd.Series(features)   #returning the dictionary as a pandas series for applying the function cleanly

feature_df = df.apply(lambda row: extract_lexicon_features(row, lexicons), axis=1)

print("\nFeature shape:", feature_df.shape)
print(feature_df.head())

# merge for one full dataframe
full_df = pd.concat([df[["label", "subject", "body"]], feature_df], axis=1)


Feature shape: (32652, 126)
   urgency_subject_raw  urgency_subject_bin  urgency_subject_norm  \
0                  0.0                  0.0                   0.0   
1                  0.0                  0.0                   0.0   
2                  0.0                  0.0                   0.0   
3                  0.0                  0.0                   0.0   
4                  0.0                  0.0                   0.0   

   urgency_body_raw  urgency_body_bin  urgency_body_norm  \
0               1.0               1.0           0.021739   
1               0.0               0.0           0.000000   
2               0.0               0.0           0.000000   
3               0.0               0.0           0.000000   
4               0.0               0.0           0.000000   

   urgency_combined_raw  urgency_combined_bin  urgency_combined_norm  \
0                   1.0                   1.0               0.019231   
1                   0.0                   0.0      

In [ ]:
# checking whether any feature is always zero
zero_only_cols = [col for col in feature_df.columns if feature_df[col].sum() == 0]
print(f"\nFeatures always zero: {len(zero_only_cols)}")
if zero_only_cols:
    print(zero_only_cols[:20])

#  summary of features
print("\nTop average feature values:")
print(feature_df.mean().sort_values(ascending=False).head(20))


Features always zero: 0

Top average feature values:
technical_discussion_combined_raw         1.270642
technical_discussion_body_raw             1.200845
request_action_combined_raw               0.748806
request_action_body_raw                   0.725499
urgency_combined_raw                      0.521193
information_transactional_combined_raw    0.519080
urgency_body_raw                          0.500092
information_transactional_body_raw        0.499510
promotion_marketing_combined_raw          0.438319
promotion_marketing_body_raw              0.408520
technical_discussion_combined_bin         0.352413
authority_combined_raw                    0.350882
authority_body_raw                        0.341020
technical_discussion_body_bin             0.337743
fear_combined_raw                         0.330056
neutral_formal_combined_raw               0.309690
fear_body_raw                             0.308220
neutral_formal_body_raw                   0.305127
request_action_combined_bin 

In [ ]:
# This is for the leakage test. I wanted to check if the features/categories were leaking data and somehow actually becoming phishing 
# predictors rather than strict intent/emotion labels

# I used the normalized values of category from the combined header+body as it is the best and most accurate representation
combined_norm_cols = [c for c in feature_df.columns if c.endswith("_combined_norm")]

X = feature_df[combined_norm_cols].copy()
y = df["label"].copy()

# train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# using logistic regression 
model = LogisticRegression(
    max_iter=2000,
    class_weight="balanced"
)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)
cm = confusion_matrix(y_test, y_pred)

print("\n Leakage Test Results")
print("Accuracy:", round(acc, 4))
print("ROC-AUC :", round(auc, 4))
print("Confusion Matrix:\n", cm)
print("\nClassification Report:\n", classification_report(y_test, y_pred, digits=4))


================ LEAKAGE TEST RESULTS ================
Accuracy: 0.7771
ROC-AUC : 0.8817
Confusion Matrix:
 [[2152 1264]
 [ 192 2923]]

Classification Report:
               precision    recall  f1-score   support

           0     0.9181    0.6300    0.7472      3416
           1     0.6981    0.9384    0.8006      3115

    accuracy                         0.7771      6531
   macro avg     0.8081    0.7842    0.7739      6531
weighted avg     0.8132    0.7771    0.7727      6531



In [30]:

# checking feature importance for the prediction of malicious vs legit


coef_df = pd.DataFrame({
    "feature": X.columns,
    "coefficient": model.coef_[0]
}).sort_values("coefficient", ascending=False)

print("\nTop positive coefficients (push toward malicious/spam):")
print(coef_df.head(5))

print("\nTop negative coefficients (push toward legitimate):")
print(coef_df.tail(5))


Top positive coefficients (push toward malicious/spam):
                              feature  coefficient
3   promotion_marketing_combined_norm    13.211222
12      sexual_pleasure_combined_norm    11.570215
0               urgency_combined_norm    10.342742
11           excitement_combined_norm     9.965803
10                trust_combined_norm     9.246576

Top negative coefficients (push toward legitimate):
                               feature  coefficient
1              authority_combined_norm    -5.443197
2         request_action_combined_norm    -7.023123
13        neutral_formal_combined_norm    -8.060822
7                   fear_combined_norm   -10.114771
6   technical_discussion_combined_norm   -37.741553


In [ ]:
#defining a helper function for using both the full feature set vs a cleaner feature set


def run_lr_model(X, y, model_name="Model"):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    model = LogisticRegression(
        max_iter=3000,
        class_weight="balanced"
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    cm = confusion_matrix(y_test, y_pred)
    report = classification_report(y_test, y_pred, digits=4)

    coef_df = pd.DataFrame({
        "feature": X.columns,
        "coefficient": model.coef_[0]
    }).sort_values("coefficient", ascending=False).reset_index(drop=True)

    print(f"\n {model_name} results ")
    print("Accuracy :", round(acc, 4))
    print("ROC-AUC  :", round(auc, 4))
    print("Confusion Matrix:\n", cm)
    print("\nClassification Report:\n", report)

    print("\nTop positive coefficients (push toward malicious/spam):")
    print(coef_df.head(10))

    print("\nTop negative coefficients (push toward legitimate):")
    print(coef_df.sort_values('coefficient', ascending=True).head(10))

    return {
        "model": model,
        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "y_pred": y_pred,
        "y_prob": y_prob,
        "accuracy": acc,
        "auc": auc,
        "coef_df": coef_df
    }


# Model with the full lexicon, uses all combined normalized categories
X_full = feature_df[combined_norm_cols].copy()
y = df["label"].copy()

results_full = run_lr_model(X_full, y, model_name="Model with full lexicon")




================ MODEL A (FULL LEXICON) RESULTS ================
Accuracy : 0.7771
ROC-AUC  : 0.8817
Confusion Matrix:
 [[2152 1264]
 [ 192 2923]]

Classification Report:
               precision    recall  f1-score   support

           0     0.9181    0.6300    0.7472      3416
           1     0.6981    0.9384    0.8006      3115

    accuracy                         0.7771      6531
   macro avg     0.8081    0.7842    0.7739      6531
weighted avg     0.8132    0.7771    0.7727      6531


Top positive coefficients (push toward malicious/spam):
                                   feature  coefficient
0        promotion_marketing_combined_norm    13.211222
1            sexual_pleasure_combined_norm    11.570215
2                    urgency_combined_norm    10.342742
3                 excitement_combined_norm     9.965803
4                      trust_combined_norm     9.246576
5                  curiosity_combined_norm     2.138043
6          social_connection_combined_norm     1.31

In [ ]:
# Model with a cleaner set of lexicons

# I dropped these features as they were giving very strong signaling toward prediction of spam or not (extremely high coefs), while 
# being very niche and specific
drop_features = [
    "technical_discussion_combined_norm",
    "sexual_pleasure_combined_norm",
    "promotion_marketing_combined_norm"
]

clean_feature_cols = [c for c in combined_norm_cols if c not in drop_features]
X_clean = feature_df[clean_feature_cols].copy()

print("\nClean feature columns:")
print(clean_feature_cols)

results_clean = run_lr_model(X_clean, y, model_name="Model with cleaner lexicon")



Clean feature columns:
['urgency_combined_norm', 'authority_combined_norm', 'request_action_combined_norm', 'social_connection_combined_norm', 'information_transactional_combined_norm', 'fear_combined_norm', 'greed_combined_norm', 'curiosity_combined_norm', 'trust_combined_norm', 'excitement_combined_norm', 'neutral_formal_combined_norm']

================ MODEL B (CLEANED FEATURES) RESULTS ================
Accuracy : 0.7203
ROC-AUC  : 0.7441
Confusion Matrix:
 [[2106 1310]
 [ 517 2598]]

Classification Report:
               precision    recall  f1-score   support

           0     0.8029    0.6165    0.6975      3416
           1     0.6648    0.8340    0.7399      3115

    accuracy                         0.7203      6531
   macro avg     0.7338    0.7253    0.7187      6531
weighted avg     0.7370    0.7203    0.7177      6531


Top positive coefficients (push toward malicious/spam):
                                   feature  coefficient
0                    urgency_combined_nor

In [28]:
from scipy.stats import mannwhitneyu

results = []

for col in clean_feature_cols:
    mal = feature_df[df["label"] == 1][col]
    legit = feature_df[df["label"] == 0][col]
    
    stat, p = mannwhitneyu(mal, legit, alternative="two-sided")
    
    results.append({
        "feature": col,
        "malicious_mean": mal.mean(),
        "legit_mean": legit.mean(),
        "p_value": p
    })

stats_df = pd.DataFrame(results).sort_values("p_value")

print(stats_df)

                                    feature  malicious_mean  legit_mean  \
1                   authority_combined_norm        0.001061    0.002092   
2              request_action_combined_norm        0.003146    0.005309   
5                        fear_combined_norm        0.000378    0.002789   
4   information_transactional_combined_norm        0.001930    0.002901   
10             neutral_formal_combined_norm        0.001242    0.003755   
3           social_connection_combined_norm        0.000975    0.000612   
9                  excitement_combined_norm        0.002937    0.000446   
6                       greed_combined_norm        0.001402    0.001486   
8                       trust_combined_norm        0.002543    0.000366   
7                   curiosity_combined_norm        0.000853    0.000295   
0                     urgency_combined_norm        0.006254    0.002585   

          p_value  
1    0.000000e+00  
2    0.000000e+00  
5    0.000000e+00  
4    0.000000e+00  

In [29]:
stats_df["effect"] = stats_df["malicious_mean"] - stats_df["legit_mean"]
print(stats_df.sort_values("effect", ascending=False))

                                    feature  malicious_mean  legit_mean  \
0                     urgency_combined_norm        0.006254    0.002585   
9                  excitement_combined_norm        0.002937    0.000446   
8                       trust_combined_norm        0.002543    0.000366   
7                   curiosity_combined_norm        0.000853    0.000295   
3           social_connection_combined_norm        0.000975    0.000612   
6                       greed_combined_norm        0.001402    0.001486   
4   information_transactional_combined_norm        0.001930    0.002901   
1                   authority_combined_norm        0.001061    0.002092   
2              request_action_combined_norm        0.003146    0.005309   
5                        fear_combined_norm        0.000378    0.002789   
10             neutral_formal_combined_norm        0.001242    0.003755   

          p_value    effect  
0    2.114960e-01  0.003669  
9    4.838205e-96  0.002491  
8    3.21